# SLAM-ASR: An Embarrassingly Simple Approach for LLM with Strong ASR Capacity

[![arXiv](https://img.shields.io/badge/arXiv-2402.08846-b31b1b.svg)](https://arxiv.org/abs/2402.08846)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gokhanersoy/slam-asr-reimplement/blob/main/slam_asr_colab.ipynb)

Bu notebook, **arXiv:2402.08846** çalışmasının Google Colab (ücretsiz T4 veya A100 GPU) üzerinde uçtan uca çalıştırılması için hazırlanmıştır.

### 🌟 Temel Mimari Prensipleri
1. **Speech Foundation Encoder (FROZEN):** HuBERT, WavLM veya Whisper ağırlıkları dondurulur.
2. **Frame Downsampler ($k=5$):** 50 Hz olan konuşma öznitelikleri, ardışık 5 çerçevenin birleştirilmesiyle **10 Hz'e** indirgenir.
3. **Trainable Projector (TRAINABLE):** Yalnızca tek katmanlı lineer katman (veya 2 katmanlı MLP) eğitilir (~18M - 21M parametre).
4. **Causal LLM Backbone (FROZEN):** Vicuna-7B veya TinyLlama dondurulmuş olarak kullanılır.
5. **Format:** `USER: <S> <Prompt> ASSISTANT: <Transcript><eos>` (Kayıp yalnızca hedef metin tokenları üzerinden hesaplanır).

## 1. GPU ve Donanım Kontrolü

In [ ]:
# GPU kullanılabilirliğini kontrol edin (Colab'da Runtime -> Change runtime type -> T4 GPU seçili olmalıdır)
!nvidia-smi
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    print(f"Device Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 2. Proje Dosyalarını Çekme ve Bağımlılıkları Yükleme

Proje dosyalarını Colab'a aktarmak için aşağıdaki **iki yöntemden birini** kullanabilirsiniz:

- **Yöntem 1 (Önerilen):** GitHub deposundan doğrudan klonlamak (`git clone`).
- **Yöntem 2:** Bilgisayarınızdaki `slam-asr-reimplement.zip` dosyasını Colab'ın sol dosya paneline sürükleyip açmak (`unzip`).

In [ ]:
import os

# Eğer çalışma dizininde slam_asr klasörü yoksa GitHub'dan klonla
if not os.path.exists("slam_asr") and not os.path.exists("slam-asr-reimplement/slam_asr"):
    print("[*] Proje GitHub'dan klonlanıyor...")
    !git clone https://github.com/gokhanersoy/slam-asr-reimplement.git
    %cd slam-asr-reimplement
elif os.path.exists("slam-asr-reimplement"):
    %cd slam-asr-reimplement
else:
    print("[*] Proje dosyaları zaten aktif dizinde mevcut.")

# Gerekli kütüphaneleri requirements.txt üzerinden yükleyin
!pip install -q -r requirements.txt

## 3. Kod Dosyalarının Doğrulanması ve Birim Testler

In [ ]:
# Testleri koşturarak modüllerin (Projector, Collator, Metrics, Model) doğru çalıştığını teyit edin
!python run_tests.py

## 4. Hızlı Doğrulama: Sentetik Veri ile Test Eğitimi
Sistemin forward-backward döngüsünün, gradyan izolasyonunun ve erken durdurma mekanizmasının çalıştığını saniyeler içinde görmek için sentetik ses verisi üretip test edebilirsiniz.

In [ ]:
# 1. Sentetik toy verisini üretin
!python scripts/create_toy_dataset.py

# 2. Hızlı debug eğitimi başlatın (CPU/T4 üzerinde anında çalışır)
!python train.py --config config/train_toy_debug.yaml --max_steps 20

## 5. Gerçek LibriSpeech Verisi Hazırlığı (Hugging Face Üzerinden)
Hugging Face `librispeech_asr` veri setinden küçük bir dilim indirip manifestoları otomatik hazırlayalım.

In [ ]:
import os
import json
from datasets import load_dataset
from tqdm import tqdm

os.makedirs("data", exist_ok=True)

def prepare_hf_split(split_name, output_path, max_samples=500):
    print(f"[*] Downloading '{split_name}' (max {max_samples} samples)...")
    ds = load_dataset("librispeech_asr", "clean", split=split_name, streaming=True)
    records = []
    for i, item in enumerate(tqdm(ds, total=max_samples)):
        if i >= max_samples:
            break
        records.append({
            "key": item["id"],
            "audio": item["audio"],  # Contains 'array' and 'sampling_rate'
            "target": item["text"],
            "prompt": "Transcribe speech to text."
        })
    with open(output_path, "w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r) + "\n")
    print(f"[*] Saved {len(records)} samples to {output_path}")

# Eğitim için 300, doğrulama için 50 örnek hazırlayalım
prepare_hf_split("train.100", "data/colab_train.jsonl", max_samples=300)
prepare_hf_split("validation", "data/colab_val.jsonl", max_samples=50)

## 6. Colab T4 Üzerinde SLAM-ASR Eğitimi

İki seçenek sunulmuştur:
1. **TinyLlama-1.1B + HuBERT-Base (`config/colab_t4_tinyllama.yaml`):** Ultra hafif (~3.8 GB VRAM), Colab T4 üzerinde son derece hızlı eğitilir.
2. **Vicuna-7B (4-Bit NF4) + HuBERT-Large (`config/colab_t4_vicuna_4bit.yaml`):** Makaledeki 7B chat modelini 4-bit kuantizasyonla T4 VRAM'ine sığdırır (~6.5 GB VRAM).

In [ ]:
# Konfigürasyonu güncelleyerek Colab verisiyle eğitimi başlatın
# Seçenek 1: TinyLlama ile hızlı eğitim (Önerilen test başlangıcı)
!python train.py \
    --config config/colab_t4_tinyllama.yaml \
    --output_dir ./outputs/slam_colab_run \
    --max_steps 300 \
    --batch_size 4

## 7. Model Değerlendirmesi (WER ve CER Hesaplama)

In [ ]:
# En iyi checkpoint üzerinde test manifestosunu değerlendirin
!python evaluate.py \
    --config config/colab_t4_tinyllama.yaml \
    --checkpoint outputs/slam_colab_run/best_checkpoint \
    --manifest data/colab_val.jsonl \
    --beam_size 4 \
    --batch_size 4

## 8. İnteraktif Çıkarım (Ses Dinleme ve Transcribe)

In [ ]:
import IPython.display as ipd
import soundfile as sf

# Örnek bir ses dosyasını dinleyin
audio_path = "data/toy_audio/toy_1.wav"
ipd.display(ipd.Audio(audio_path))

# Modeli kullanarak sesi yazıya dökün
!python inference.py \
    --audio data/toy_audio/toy_1.wav \
    --config config/colab_t4_tinyllama.yaml \
    --checkpoint outputs/slam_colab_run/best_checkpoint/projector.pt \
    --prompt "Transcribe speech to text." \
    --beam_size 4